In [ ]:
# NOTEBOOK NAME
# CustomTrackerSandbox.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

# # for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize
import matplotlib.colors as mcolors

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

In [ ]:
# COMBINE ALL OF THE DATA FRAMES FROM EACH TIME AT ONE LEVEL INTO ONE BIG DATA FRAME

# CHOOSE COMPRESSED OR UNCOMRESSED FILES
CompBool = 1     # 1 for yes compressed, 0 for no, uncompressed files

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# CHOOSE YOUR ALTITUDE
Altitude = 2000  # [m] choose a multiple of 500 m to look at a CAPI for

# CHOOSE YOUR VARIABLE
Var = 'Z'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# CHOOSE YOUR QUALITY CONTROL SETTINGS
# taken from Aragon et al. 2024
MinValidZDR = -4 # NO VALID DATA TO USE THE OPTION YET
MaxValidZDR =  4 # NO VALID DATA TO USE THE OPTION YET
MinValidRhoHV = 0.85





# USER CHOICE FOLLOW-ON SECTION

# compression choice follow-on (select the folder to load from)
if (CompBool):
    RadarGridsFolder = 'CompressedRadarGrids'
else:
    RadarGridsFolder = 'RadarGrids'  

# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'Horz'

# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

# Initialize a list to store time slices
time_slices = []
time_coords = []

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)

    NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                             + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
    # try to load in the netcdf file and if it doesn't work, just keep going through the loop
    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # index in the netcdf altitude variable for the altitude you want
    alti = np.where(xgrid.z == Altitude) # this is a double nested array for some reason
    alti = alti[0][0] # take the index out of the double nested array

    # quit out if the altitude does not correspond to one in the netCDF file
    if (np.size(alti) != 1): 
        raise ValueError(str(Altitude) + ' m is not a valid altitude in the data')

    # Extract corrected_reflectivity at this altitude and time
    # Shape: (301, 301, 1) — we'll squeeze out the time dimension later
    reflectivity_slice = xgrid['corrected_reflectivity'].isel(z=alti, time=0)
    
    time_slices.append(reflectivity_slice)
    time_coords.append(xgrid.time.values[0])

# Concatenate all time slices along the time dimension
xgrid_combined = xr.concat(time_slices, dim='time')

# Assign the correct time coordinates
xgrid_combined = xgrid_combined.assign_coords(time=time_coords)

# Rename to preserve the variable name
xgrid_combined = xgrid_combined.to_dataset(name='corrected_reflectivity')

print(f'Combined xarray shape: {xgrid_combined["corrected_reflectivity"].shape}')
# Expected output: (301, 301, 288)

In [ ]:
xgrid_combined['corrected_reflectivity']

In [ ]:
# OVER TIME TRACKING VERSION

import numpy as np
from scipy.ndimage import label
import pandas as pd

# ── helpers ───────────────────────────────────────────────────────────────────

def _km_offset_to_latlon(centre_x_km, centre_y_km, radar_lat, radar_lon):
    lat = radar_lat + (centre_y_km / 111.32)
    lon = radar_lon + (centre_x_km / (111.32 * np.cos(np.radians(radar_lat))))
    return lat, lon


def _detect_features(refl, x_coords, y_coords, threshold_dbz, cell_area_km2,
                     radar_lat, radar_lon):
    refl = np.squeeze(refl)

    if refl.ndim != 2:
        raise ValueError(
            f"Expected a 2D reflectivity slice after squeezing, "
            f"but got shape {refl.shape}. Check your time indexing."
        )

    mask            = refl > threshold_dbz
    structure_8conn = np.ones((3, 3), dtype=int)
    labelled_grid, n_features = label(mask, structure=structure_8conn)

    x_grid_2d, y_grid_2d = np.meshgrid(x_coords, y_coords)

    features = {}
    for fid in range(1, n_features + 1):
        fmask     = labelled_grid == fid
        refl_vals = refl[fmask]
        linear_z  = 10.0 ** (refl_vals / 10.0)
        total_w   = linear_z.sum()
        n_cells   = int(fmask.sum())

        centre_x = float((x_grid_2d[fmask] * linear_z).sum() / total_w)
        centre_y = float((y_grid_2d[fmask] * linear_z).sum() / total_w)

        centre_lat, centre_lon = _km_offset_to_latlon(
            centre_x, centre_y, radar_lat, radar_lon
        )

        features[fid] = {
            'local_id'         : fid,
            'n_cells'          : n_cells,
            'area_km2'         : float(n_cells * cell_area_km2),
            'mean_dbz'         : float(refl_vals.mean()),
            'centre_x_km'      : centre_x,
            'centre_y_km'      : centre_y,
            'centre_lat'       : round(centre_lat, 5),
            'centre_lon'       : round(centre_lon, 5),
            'refl_mass_dBZkm2' : float(10.0 * np.log10((linear_z * cell_area_km2).sum())),
            'mask'             : fmask,
        }

    return features, labelled_grid


def _compute_overlap_links(old_features, new_features, overlap_threshold=0.20):
    old_to_new = {oid: [] for oid in old_features}
    new_to_old = {nid: [] for nid in new_features}

    for oid, of in old_features.items():
        for nid, nf in new_features.items():
            intersection = int((of['mask'] & nf['mask']).sum())
            if intersection == 0:
                continue
            larger_size = max(of['n_cells'], nf['n_cells'])
            fraction    = intersection / larger_size
            if fraction > overlap_threshold:
                old_to_new[oid].append(nid)
                new_to_old[nid].append(oid)

    return old_to_new, new_to_old


# ── main tracker ──────────────────────────────────────────────────────────────

def track_reflectivity_features(
    xgrid,
    threshold_dbz     = 20.0,
    overlap_threshold = 0.20,
):
    """
    Track radar reflectivity features across all time steps in xgrid.

    Parameters
    ----------
    xgrid : xarray.Dataset
        Expected dimensions: (time, y, x).
        Must contain 'origin_latitude' and 'origin_longitude'.
    threshold_dbz : float
    overlap_threshold : float

    Returns
    -------
    results : dict  { timestamp : pd.DataFrame }
    all_labels : dict { timestamp : np.ndarray }
    """

    # ── coordinate setup ──────────────────────────────────────────────────────
    x_coords      = xgrid['x'].values / 1000.0
    y_coords      = xgrid['y'].values / 1000.0
    cell_area_km2 = (x_coords[1] - x_coords[0]) * (y_coords[1] - y_coords[0])
    timestamps    = xgrid['time'].values
    n_times       = len(timestamps)

    radar_lat = float(xgrid['origin_latitude'].values)
    radar_lon = float(xgrid['origin_longitude'].values)

    print(f"Radar origin: {radar_lat:.4f}°N, {radar_lon:.4f}°E")

    # ── global state ──────────────────────────────────────────────────────────
    next_global_id       = 1
    results              = {}
    all_labels           = {}
    prev_global_features = {}
    prev_local_to_global = {}
    prev_features_raw    = {}
    prev_timestamp       = None

    # Master history store — keyed by global feature ID
    # Accumulates ALL events for a feature across its entire lifetime
    # Pruned when a feature dies to free memory
    master_history = {}   # { global_id : [event_dict, ...] }

    # ── iterate over timesteps ────────────────────────────────────────────────
    for t_idx in range(n_times):
        timestamp = timestamps[t_idx]
        refl      = xgrid['corrected_reflectivity'][t_idx].values

        new_features_raw, labelled_grid = _detect_features(
            refl, x_coords, y_coords, threshold_dbz, cell_area_km2,
            radar_lat, radar_lon
        )

        new_local_to_global = {}

        motion_info = {
            nid: {
                'u'            : np.nan,
                'v'            : np.nan,
                'prev_mass'    : np.nan,
                'motion_clean' : np.nan,
            }
            for nid in new_features_raw
        }

        # ── first frame: everything is a birth ────────────────────────────────
        if t_idx == 0:
            for nid in new_features_raw:
                gid                      = next_global_id
                next_global_id          += 1
                new_local_to_global[nid] = gid
                master_history[gid]      = [{'t': timestamp, 'event': 'born'}]

        # ── subsequent frames: full tracking logic ────────────────────────────
        else:
            dt_seconds = float(
                (pd.Timestamp(timestamp) - pd.Timestamp(prev_timestamp))
                .total_seconds()
            )

            old_to_new, new_to_old = _compute_overlap_links(
                prev_features_raw, new_features_raw, overlap_threshold
            )

            assigned_new        = {}   # { new_local_id : global_id }
            split_or_merge_nids = set()

            # Keep track of which old global IDs have been consumed as merge
            # losers so we don't also mark them as dead in Step 5
            merge_loser_gids = set()

            # ── STEP 2: resolve splits ────────────────────────────────────────
            # An old feature that maps to MORE THAN ONE new feature
            for oid, linked_nids in old_to_new.items():
                if len(linked_nids) <= 1:
                    continue

                old_gid = prev_local_to_global[oid]

                linked_nids_sorted = sorted(
                    linked_nids,
                    key=lambda nid: new_features_raw[nid]['refl_mass_dBZkm2'],
                    reverse=True,
                )

                # Flag ALL new features from this split as contaminated
                for nid in linked_nids_sorted:
                    split_or_merge_nids.add(nid)

                winner_nid = linked_nids_sorted[0]
                loser_nids = linked_nids_sorted[1:]

                # Assign loser IDs first so we can reference them in the
                # winner's history entry
                loser_gids = []
                for loser_nid in loser_nids:
                    if loser_nid not in assigned_new:
                        new_gid                 = next_global_id
                        next_global_id         += 1
                        assigned_new[loser_nid] = new_gid
                        master_history[new_gid] = [{
                            't'     : timestamp,
                            'event' : 'split_from',
                            'id'    : old_gid,
                        }]
                    loser_gids.append(assigned_new[loser_nid])

                # Winner inherits old ID — only write history once
                if winner_nid not in assigned_new:
                    assigned_new[winner_nid] = old_gid
                    master_history[old_gid].append({
                        't'     : timestamp,
                        'event' : 'split_winner',
                        'ids'   : loser_gids,
                    })

            # ── STEP 3: simple continuation candidates ────────────────────────
            # Only note the candidate — do NOT assign yet so Step 4 can see
            # all competing old IDs for every new feature before deciding
            continuation_candidates = {}  # { new_local_id : old_global_id }
            for oid, linked_nids in old_to_new.items():
                if len(linked_nids) != 1:
                    continue
                nid     = linked_nids[0]
                old_gid = prev_local_to_global[oid]
                # Only register if not already claimed by a split
                if nid not in assigned_new:
                    continuation_candidates[nid] = old_gid

            # ── STEP 4: resolve merges and continuations together ─────────────
            for nid in new_features_raw:
                linked_oids = new_to_old.get(nid, [])

                # Gather every old global ID that has a valid overlap link
                # to this new feature, using the OLD feature's mass throughout
                candidates = {}  # { old_global_id : old_refl_mass_dBZkm2 }

                for oid in linked_oids:
                    old_gid = prev_local_to_global[oid]
                    # Always use the previous frame's mass for fair comparison
                    candidates[old_gid] = prev_global_features[old_gid]['refl_mass_dBZkm2']

                # Also include any split-winner already assigned to this nid,
                # again using the OLD feature's mass
                if nid in assigned_new:
                    existing_gid = assigned_new[nid]
                    if existing_gid not in candidates and existing_gid in prev_global_features:
                        candidates[existing_gid] = prev_global_features[existing_gid]['refl_mass_dBZkm2']

                # Also include simple continuation candidates
                if nid in continuation_candidates:
                    cont_gid = continuation_candidates[nid]
                    if cont_gid not in candidates:
                        candidates[cont_gid] = prev_global_features[cont_gid]['refl_mass_dBZkm2']

                if len(candidates) == 0:
                    # No old feature links — birth handled in Step 5
                    continue

                elif len(candidates) == 1:
                    winning_gid = list(candidates.keys())[0]

                    # FIX Bug 3: do NOT overwrite a split loser's brand new ID
                    if nid not in assigned_new:
                        assigned_new[nid] = winning_gid
                    elif assigned_new[nid] != winning_gid:
                        # nid was assigned as a split loser — leave it alone
                        pass

                else:
                    # Genuine merge — flag and pick winner by OLD mass
                    split_or_merge_nids.add(nid)
                    winning_gid = max(candidates, key=candidates.get)

                    # FIX Bug 3: only assign if not already a split loser
                    if nid not in assigned_new or assigned_new[nid] in prev_global_features:
                        assigned_new[nid] = winning_gid

                    loser_gids = [gid for gid in candidates if gid != winning_gid]

                    for gid in loser_gids:
                        merge_loser_gids.add(gid)

                        # Append merged_into to loser's master history
                        if gid in master_history:
                            master_history[gid].append({
                                't'     : timestamp,
                                'event' : 'merged_into',
                                'id'    : winning_gid,
                            })

                        # FIX Bug 5: also write died for merge losers
                        master_history[gid].append({
                            't'     : timestamp,
                            'event' : 'died',
                        })

                    # Winner records all absorbed IDs in one event
                    master_history[winning_gid].append({
                        't'     : timestamp,
                        'event' : 'absorbed',
                        'ids'   : loser_gids,
                    })

            # ── STEP 5: births and deaths ─────────────────────────────────────

            # Deaths — old features with no links at all
            # Exclude merge losers as they already got their died event above
            for oid, linked_nids in old_to_new.items():
                if len(linked_nids) == 0:
                    old_gid = prev_local_to_global[oid]
                    if old_gid not in merge_loser_gids:
                        master_history[old_gid].append({
                            't'     : timestamp,
                            'event' : 'died',
                        })

            # Births — new features with no assignment at all
            for nid in new_features_raw:
                if nid not in assigned_new:
                    new_gid             = next_global_id
                    next_global_id     += 1
                    assigned_new[nid]   = new_gid
                    master_history[new_gid] = [{'t': timestamp, 'event': 'born'}]

            new_local_to_global = assigned_new

            # ── STEP 6: compute motion info ───────────────────────────────────
            for nid, gid in new_local_to_global.items():

                # Find which old local ID this global ID came from (if any)
                prev_oid = None
                for oid, old_gid in prev_local_to_global.items():
                    if old_gid == gid:
                        prev_oid = oid
                        break

                if prev_oid is None:
                    # Born or split loser with brand new ID — no previous position
                    continue

                prev_f = prev_features_raw[prev_oid]
                curr_f = new_features_raw[nid]

                dx_m = (curr_f['centre_x_km'] - prev_f['centre_x_km']) * 1000.0
                dy_m = (curr_f['centre_y_km'] - prev_f['centre_y_km']) * 1000.0

                motion_info[nid]['u']            = dx_m / dt_seconds
                motion_info[nid]['v']            = dy_m / dt_seconds
                motion_info[nid]['prev_mass']    = prev_f['refl_mass_dBZkm2']
                motion_info[nid]['motion_clean'] = (
                    False if nid in split_or_merge_nids else True
                )

        # ── build global labelled grid ─────────────────────────────────────────
        global_labelled = np.zeros_like(labelled_grid)
        for nid, gid in new_local_to_global.items():
            global_labelled[new_features_raw[nid]['mask']] = gid
        all_labels[timestamp] = global_labelled

        # ── build output DataFrame ─────────────────────────────────────────────
        rows = []
        for nid, gid in new_local_to_global.items():
            f  = new_features_raw[nid]
            mi = motion_info[nid]
            rows.append({
                'feature_id'       : gid,
                'n_cells'          : f['n_cells'],
                'area_km2'         : round(f['area_km2'],         2),
                'mean_dbz'         : round(f['mean_dbz'],         2),
                'centre_x_km'      : round(f['centre_x_km'],      3),
                'centre_y_km'      : round(f['centre_y_km'],      3),
                'centre_lat'       : f['centre_lat'],
                'centre_lon'       : f['centre_lon'],
                'refl_mass_dBZkm2' : round(f['refl_mass_dBZkm2'], 2),
                'u_ms'             : mi['u'],
                'v_ms'             : mi['v'],
                'prev_mass_dBZkm2' : mi['prev_mass'],
                'motion_clean'     : mi['motion_clean'],
                # Snapshot of full accumulated history at this point in time
                'history'          : list(master_history.get(gid, [])),
            })

        if len(rows) == 0:
            df = pd.DataFrame(columns=[
                'feature_id', 'n_cells', 'area_km2', 'mean_dbz',
                'centre_x_km', 'centre_y_km', 'centre_lat', 'centre_lon',
                'refl_mass_dBZkm2', 'u_ms', 'v_ms', 'prev_mass_dBZkm2',
                'motion_clean', 'history'
            ])
        else:
            df = pd.DataFrame(rows).sort_values('feature_id').reset_index(drop=True)

        results[timestamp] = df

        print(f"t={timestamp} — {len(rows)} feature(s)")
        if len(rows) > 0:
            print(df.drop(columns='history').to_string(index=False))
        print()

        # ── update previous-frame state ───────────────────────────────────────
        prev_global_features = {}
        for nid, gid in new_local_to_global.items():
            f = {k: v for k, v in new_features_raw[nid].items() if k != 'mask'}
            f['history'] = list(master_history.get(gid, []))
            prev_global_features[gid] = f

        prev_local_to_global = new_local_to_global
        prev_features_raw    = new_features_raw
        prev_timestamp       = timestamp

        # FIX Bug 6: prune master_history for features that are now dead
        # Keep a set of all currently live global IDs
        live_gids = set(new_local_to_global.values())
        dead_gids = [gid for gid in master_history if gid not in live_gids]
        for gid in dead_gids:
            del master_history[gid]

    return results, all_labels


In [ ]:
# ACTUALLY USE THE FUNCTION

FeatureDBZmin = 30.0

results, all_labels = track_reflectivity_features(
    xgrid_combined,
    threshold_dbz     = FeatureDBZmin,
    overlap_threshold = 0.20)

In [ ]:
# CHAD FUNCTION FOR BULK STATISTICS ON ALL FEATURES IN THE DAY

def summarise_features(results):
    """
    Summarise the full lifetime of every feature tracked across all timesteps.

    One row per feature containing birth/death info, mean motion, mean size
    and mean reflectivity mass.

    Parameters
    ----------
    results : dict { timestamp : pd.DataFrame }
        Output from track_reflectivity_features().

    Returns
    -------
    summary : pd.DataFrame
        One row per feature.
    """

    timestamps     = list(results.keys())
    first_ts       = timestamps[0]
    last_ts        = timestamps[-1]

    # ── collect per-feature per-timestep records ──────────────────────────────
    # feature_records[gid] = list of dicts, one per timestep the feature exists
    feature_records = {}   # { global_id : [{'t', 'x', 'y', 'lat', 'lon',
                           #                  'area_km2', 'refl_mass', 'history'}, ...] }

    for ts, df in results.items():
        if df.empty:
            continue
        for _, row in df.iterrows():
            gid = row['feature_id']
            if gid not in feature_records:
                feature_records[gid] = []
            feature_records[gid].append({
                't'          : ts,
                'x_km'       : row['centre_x_km'],
                'y_km'       : row['centre_y_km'],
                'lat'        : row['centre_lat'],
                'lon'        : row['centre_lon'],
                'area_km2'   : row['area_km2'],
                'refl_mass'  : row['refl_mass_dBZkm2'],
                'history'    : row['history'],
            })

    # ── build one summary row per feature ─────────────────────────────────────
    rows = []

    for gid, records in feature_records.items():
        # Sort by time just in case
        records = sorted(records, key=lambda r: r['t'])

        birth_rec = records[0]
        death_rec = records[-1]

        birth_t   = birth_rec['t']
        death_t   = death_rec['t']

        # Time alive in seconds
        dt_alive  = float(
            (pd.Timestamp(death_t) - pd.Timestamp(birth_t)).total_seconds()
        )

        # ── mean U and V from bulk displacement ───────────────────────────────
        if dt_alive > 0:
            dx_m  = (death_rec['x_km'] - birth_rec['x_km']) * 1000.0
            dy_m  = (death_rec['y_km'] - birth_rec['y_km']) * 1000.0
            mean_u = dx_m / dt_alive
            mean_v = dy_m / dt_alive
        else:
            mean_u = np.nan
            mean_v = np.nan

        # ── mean size and mass over lifetime ──────────────────────────────────
        mean_area      = float(np.mean([r['area_km2']  for r in records]))
        mean_refl_mass = float(np.mean([r['refl_mass'] for r in records]))

        # ── parse history for boolean flags ───────────────────────────────────
        # Use the most complete history — from the last record
        history = death_rec['history']

        event_types = [e['event'] for e in history]

        started_in_split  = 'split_from'   in event_types
        ended_in_merge    = 'merged_into'  in event_types
        alive_at_first    = birth_t == first_ts
        alive_at_last     = death_t == last_ts

        rows.append({
            'feature_id'        : gid,
            'birth_t'           : birth_t,
            'death_t'           : death_t,
            'time_alive_s'      : dt_alive,
            'birth_x_km'        : round(birth_rec['x_km'],  3),
            'birth_y_km'        : round(birth_rec['y_km'],  3),
            'birth_lat'         : birth_rec['lat'],
            'birth_lon'         : birth_rec['lon'],
            'death_x_km'        : round(death_rec['x_km'],  3),
            'death_y_km'        : round(death_rec['y_km'],  3),
            'death_lat'         : death_rec['lat'],
            'death_lon'         : death_rec['lon'],
            'mean_u_ms'         : mean_u,
            'mean_v_ms'         : mean_v,
            'mean_area_km2'     : round(mean_area,      2),
            'mean_refl_mass_dBZkm2' : round(mean_refl_mass, 2),
            'n_timesteps'       : len(records),
            'started_in_split'  : started_in_split,
            'ended_in_merge'    : ended_in_merge,
            'alive_at_first_frame' : alive_at_first,
            'alive_at_last_frame'  : alive_at_last,
        })

    summary = pd.DataFrame(rows).sort_values('feature_id').reset_index(drop=True)

    print(f"Summarised {len(summary)} unique features.")
    print(summary.to_string(index=False))

    return summary



In [ ]:
summary = summarise_features(results)

In [ ]:
time_index = 229

print([timestamps[time_index]])

for i in range(0,len(results[timestamps[time_index]]['feature_id'])):
    print(results[timestamps[time_index]]['feature_id'][i])
    print(results[timestamps[time_index]]['history'][i])
    print('')

In [ ]:
# CHAD CREATED SCATTER PLOT OF FEATURE PROPOGATION DIRECTIONS

def plot_uv_scatter(results, clean_motion_only=False, cmap_name='nipy_spectral', figsize=(7, 7)):
    """
    Scatter plot of U vs V motion speeds for all features across all timesteps.
    Dots are coloured by time of day using a cyclic colourmap.

    Parameters
    ----------
    results : dict { timestamp : pd.DataFrame }
        Output from track_reflectivity_features().
    clean_motion_only : bool
        If True, only plot points where motion_clean == True.
    cmap_name : str
        Matplotlib colourmap name. Default is 'nipy_spectral'.
    figsize : tuple
        Figure size.
    """

    # ── collect all U/V pairs and timestamps across all timesteps ─────────────
    all_u          = []
    all_v          = []
    all_minutes    = []   # time of day in minutes for colouring

    for timestamp, df in results.items():
        if df.empty:
            continue

        valid = df[df['u_ms'].notna() & df['v_ms'].notna()].copy()

        if clean_motion_only:
            valid = valid[valid['motion_clean'] == True]

        if valid.empty:
            continue

        # Time of day in minutes for this timestep
        dt = pd.Timestamp(timestamp)
        minutes = dt.hour * 60 + dt.minute + dt.second / 60.0

        all_u.extend(valid['u_ms'].tolist())
        all_v.extend(valid['v_ms'].tolist())
        all_minutes.extend([minutes] * len(valid))

    all_u       = np.array(all_u)
    all_v       = np.array(all_v)
    all_minutes = np.array(all_minutes)

    print(f"Plotting {len(all_u)} U/V data points.")

    # ── plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    sc = ax.scatter(
        all_u, all_v,
        c          = all_minutes,
        cmap       = plt.get_cmap(cmap_name),
        vmin       = 0,
        vmax       = 1440,
        s          = 10,
        alpha      = 0.6,
        edgecolors = 'none',
    )

    # Zero lines for reference
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

    # Equal axes
    max_val = np.nanmax(np.abs(np.concatenate([all_u, all_v]))) * 1.1
    ax.set_xlim(-max_val, max_val)
    ax.set_ylim(-max_val, max_val)
    ax.set_aspect('equal')

    # Labels
    ax.set_xlabel('U speed (m/s)  [West ← 0 → East]')
    ax.set_ylabel('V speed (m/s)  [South ← 0 → North]')
    title_suffix = ' (clean motion only)' if clean_motion_only else ' (all motion)'
    ax.set_title(f'Feature U–V Motion Scatter{title_suffix}\nn = {len(all_u)} points')

    # ── time of day colourbar ─────────────────────────────────────────────────
    cbar = plt.colorbar(sc, ax=ax, orientation='vertical',
                        pad=0.02, fraction=0.046, shrink=1.0)
    cbar.set_label('Time of Day (UTC)')

    hourly_ticks = np.arange(0, 1441, 60)
    cbar.set_ticks(hourly_ticks)
    tick_labels = [
        f'{int(m // 60):02d}:00' if m % 180 == 0 else ''
        for m in hourly_ticks
    ]
    cbar.set_ticklabels(tick_labels)
    cbar.ax.tick_params(which='major', length=4, width=0.8)

    plt.xlim([-10,10])
    plt.ylim([-10,10])

    plt.tight_layout()
    plt.show()

    return fig, ax




In [ ]:
# CHAD WIND ROSE

results = results
clean_motion_only=True
cmap_name='plasma'
figsize=(8, 8)

min_lifetime_minutes = 17.5   # only features alive longer than this are included


# # ── collect U/V speeds and weights ───────────────────────────────────────
# all_u       = []
# all_v       = []
# all_weights = []

# for timestamp, df in results.items():
#     if df.empty:
#         continue

#     # Must have valid speed AND a valid previous mass to weight by
#     valid = df[
#         df['u_ms'].notna() &
#         df['v_ms'].notna() &
#         df['prev_mass_dBZkm2'].notna()
#     ].copy()

#     if clean_motion_only:
#         valid = valid[valid['motion_clean'] == True]

#     if valid.empty:
#         continue

#     all_u.extend(valid['u_ms'].tolist())
#     all_v.extend(valid['v_ms'].tolist())
#     all_weights.extend(valid['prev_mass_dBZkm2'].tolist())

# all_u       = np.array(all_u)
# all_v       = np.array(all_v)
# all_weights = np.array(all_weights)

# print(f"Plotting {len(all_u)} propagation vectors (weighted by prev_mass_dBZkm2).")

# SUMMARY VERSION
# SUMMARY VERSION
# SUMMARY VERSION
# SUMMARY VERSION
# ── filter summary by minimum lifetime ───────────────────────────────────
min_lifetime_seconds = min_lifetime_minutes * 60
summary_filtered     = summary[summary['time_alive_s'] > min_lifetime_seconds]

print(f"Features before lifetime filter : {len(summary)}")
print(f"Features after lifetime filter  : {len(summary_filtered)} (> {min_lifetime_minutes} min)")

# ── collect U/V speeds and weights from filtered summary ─────────────────
valid_mask  = summary_filtered['mean_u_ms'].notna() & summary_filtered['mean_v_ms'].notna()
all_u       = summary_filtered.loc[valid_mask, 'mean_u_ms'].values
all_v       = summary_filtered.loc[valid_mask, 'mean_v_ms'].values
all_weights = summary_filtered.loc[valid_mask, 'mean_area_km2'].values

print(f"Plotting {len(all_u)} feature mean propagation vectors (weighted by mean_area_km2).")
# SUMMARY VERSION END
# SUMMARY VERSION END
# SUMMARY VERSION END
# SUMMARY VERSION END

# ── convert U/V to speed and direction ────────────────────────────────────
speeds     = np.sqrt(all_u**2 + all_v**2)
directions = (np.degrees(np.arctan2(all_u, all_v))) % 360.0

# ── define bins ───────────────────────────────────────────────────────────
dir_labels  = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
n_dirs      = 8
dir_width   = 360.0 / n_dirs
dir_centres = np.arange(0, 360, dir_width)

speed_edges  = np.arange(0, 24, 2)          # [0, 2, 4, ... 20]
n_speed_bins = len(speed_edges) - 1
speed_labels = [f'{int(speed_edges[i])}–{int(speed_edges[i+1])} m/s'
                for i in range(n_speed_bins - 1)]
speed_labels.append('>20 m/s')

speeds_clipped = np.clip(speeds, 0, 22.0 - 1e-6)

# ── bin the data — sum weights instead of counting ────────────────────────
# weighted_counts[dir_idx, speed_idx] = sum of prev_mass_dBZkm2 in that bin
weighted_counts = np.zeros((n_dirs, n_speed_bins))

for i in range(n_dirs):
    centre = dir_centres[i]
    low    = (centre - dir_width / 2.0) % 360.0
    high   = (centre + dir_width / 2.0) % 360.0

    if low > high:
        dir_mask = (directions >= low) | (directions < high)
    else:
        dir_mask = (directions >= low) & (directions < high)

    dir_speeds  = speeds_clipped[dir_mask]
    dir_weights = all_weights[dir_mask]

    for j in range(n_speed_bins):
        speed_mask              = (dir_speeds >= speed_edges[j]) & (dir_speeds < speed_edges[j + 1])
        # Sum weights rather than count
        weighted_counts[i, j]  = dir_weights[speed_mask].sum()

# Convert to percentage of total weight
total_weight    = weighted_counts.sum()
counts_pct      = (weighted_counts / total_weight) * 100.0 if total_weight > 0 else weighted_counts

# ── colourmap for speed layers ─────────────────────────────────────────────
cmap        = plt.get_cmap(cmap_name)
colour_vals = np.linspace(0.0, 1.0, n_speed_bins)
colours     = [cmap(v) for v in colour_vals]

# ── plot ──────────────────────────────────────────────────────────────────
fig      = plt.figure(figsize=figsize)
ax_polar = fig.add_subplot(111, projection='polar')

ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)

bar_width = np.radians(dir_width * 0.9)
theta     = np.radians(dir_centres)

bottoms = np.zeros(n_dirs)

for j in range(n_speed_bins):
    ax_polar.bar(
        theta,
        counts_pct[:, j],
        width     = bar_width,
        bottom    = bottoms,
        color     = colours[j],
        edgecolor = 'white',
        linewidth = 0.5,
        label     = speed_labels[j],
    )
    bottoms += counts_pct[:, j]

# ── axes formatting ───────────────────────────────────────────────────────
ax_polar.set_xticks(np.radians(dir_centres))
ax_polar.set_xticklabels(dir_labels, fontsize=11)

ax_polar.set_ylabel('Weighted Frequency (%)', labelpad=30, fontsize=10)

max_pct = bottoms.max()
r_ticks = np.arange(0, max_pct + 2, max(1, round(max_pct / 5)))
ax_polar.set_yticks(r_ticks)
ax_polar.set_yticklabels([f'{r:.0f}%' for r in r_ticks], fontsize=8)
ax_polar.set_ylim(0, max_pct * 1.05)   # ← 15% headroom above the tallest bar

title_suffix = ' (clean motion only)' if clean_motion_only else ' (all motion)'
ax_polar.set_title(
    f'Feature (dBZ > {FeatureDBZmin}) Propagation Rose for {RadarSiteName} Radar\n at {Altitude*0.001} km Altitude'
    f' on {RadarFileDatePrint} UTC\n({len(all_u)} Features weighted by "Mean Area" {title_suffix})\nConsidering Features Lasting > {min_lifetime_minutes} Mins',
    pad=20, fontsize=12
)

# ── colourbar for speed — discretised to match wind rose bins ─────────────────
# Generate n_speed_bins colours for the bars + 1 extra darker colour for the
# >20 triangle so it is visually distinct from the 18-20 bin
colour_vals   = np.linspace(0.0, 1.0, n_speed_bins)
bar_colours   = [cmap(v) for v in colour_vals]
over_colour   = cmap(1.0)                        # darkest end of the colourmap

cmap_discrete = ListedColormap(bar_colours)
cmap_discrete.set_over(over_colour)              # triangle gets this distinct colour

norm = mcolors.BoundaryNorm(speed_edges, n_speed_bins)
sm   = plt.cm.ScalarMappable(cmap=cmap_discrete, norm=norm)
sm.set_array([])

cbar = plt.colorbar(sm, ax=ax_polar, orientation='vertical',
                    pad=0.1, fraction=0.03, shrink=0.7,
                    boundaries=speed_edges, spacing='uniform',
                    extend='max')
cbar.set_label('Speed (m/s)')
cbar.set_ticks(speed_edges)
cbar.set_ticklabels([f'{int(e)}' for e in speed_edges[:-1]] + [''])



plt.tight_layout()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/FeaturePropogation/' + RadarIDno + '/' + RadarFileDate + '/'
                                                                                    
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + str(int(FeatureDBZmin)) + 'DBZmin_' + str(min_lifetime_minutes) + 'MinsMin_SummaryPropogationRose_' + str(Altitude) + 'm.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
plt.close()

In [ ]:
fig, ax = plot_propagation_rose(results, clean_motion_only=True, cmap_name='YlOrRd')

SavePath = '/scratch/v46/sg3241/tmp/pngImages/FeaturePropogation/22_20240214_6000m.png'

fig.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=300)
plt.close()  # Close after saving

In [ ]:
# CHAD CREATED FEATURE TRACKING PLOTTER

def plot_feature_tracks(ax, results, linewidth=1.5):
    """
    Plot centre-of-mass tracks for features that exist for more than one frame.

    Line colour is determined by the time-of-day of the segment's departure
    point, using a cyclic colourmap (HSV, 00:00→23:59 maps to 0→1).
    Line width is uniform.

    Parameters
    ----------
    ax : cartopy GeoAxes
    results : dict { timestamp : pd.DataFrame }
        Output from track_reflectivity_features().
    linewidth : float
        Uniform line width for all segments.
    """


    # ── cyclic time-of-day colourmap ──────────────────────────────────────────
    cmap = plt.get_cmap('hsv')

    def time_to_colour(timestamp):
        """
        Convert a numpy datetime64 timestamp to a colour based on time of day.
        00:00 → 0.0, 23:59 → ~1.0
        """
        dt = pd.Timestamp(timestamp)
        minutes_since_midnight = dt.hour * 60 + dt.minute + dt.second / 60.0
        normalised = minutes_since_midnight / 1440.0
        return plt.get_cmap('nipy_spectral')(normalised)

    # ── build per-feature time series ─────────────────────────────────────────
    all_ids = set()
    for df in results.values():
        all_ids.update(df['feature_id'].tolist())

    tracks = {fid: [] for fid in all_ids}

    for timestamp, df in results.items():
        for _, row in df.iterrows():
            tracks[row['feature_id']].append((
                timestamp,
                row['centre_lon'],
                row['centre_lat'],
            ))

    # Sort each track by timestamp
    for fid in tracks:
        tracks[fid].sort(key=lambda x: x[0])

    # ── plot only features with more than one timestep ────────────────────────
    for fid, track in tracks.items():
        if len(track) < 2:
            continue

        lons       = [entry[1] for entry in track]
        lats       = [entry[2] for entry in track]
        timestamps = [entry[0] for entry in track]

        for i in range(len(track) - 1):
            colour = time_to_colour(timestamps[i])
            ax.plot(
                [lons[i], lons[i + 1]],
                [lats[i], lats[i + 1]],
                color          = colour,
                linewidth      = linewidth,
                transform      = ccrs.PlateCarree(),
                zorder         = 20,
                alpha          = 0.85,
                solid_capstyle = 'round',
            )

In [ ]:
# TRACKS ON A MAP!

fig, ax = plt.subplots(figsize=(8, 6), 
                       subplot_kw={'projection': ccrs.PlateCarree()})

# gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
# EXPERIMENTAL TOPO SHADING SECTION
# EXPERIMENTAL TOPO SHADING SECTION
# EXPERIMENTAL TOPO SHADING SECTION

# TERRAIN SHADING USING LOCAL GEBCO DEM
lon_min, lon_max = float(xgrid.lon.min()), float(xgrid.lon.max())
lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

try:
    # Path to your GEBCO NetCDF
    gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'

    # Use the helper to get a subset over the radar domain
    dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

    if dem_da is not None:
        # dem_da is an xarray.DataArray with coords lon, lat
        dem_lon = dem_da.lon.values
        dem_lat = dem_da.lat.values
        dem_data = dem_da.values

        # Make 2D lon/lat grids if necessary
        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        # Ensure we have some valid data
        valid = np.isfinite(dem_data)
        if not np.any(valid):
            raise ValueError('DEM has no finite values in this domain')

        colours = [
            '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
        
            '#c4dec2',  # 1: 0–200 m, pale green
            '#e4edc9',  # 2: 200–400 m, greenish-yellow
            '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
            '#e9d7bd',  # 4: 600–800 m, light tan
            '#ddc4aa',  # 5: 800–1000 m, tan
            '#cfb194',  # 6: 1000–1200 m, light brown
            '#b58f6e',  # 7: > 1200 m, darker brown
        ]
        
        bounds = [
            -1000.0,  # ocean below 0
            0.0,      # 0–200
            200.0,    # 200–400
            400.0,    # 400–600
            600.0,    # 600–800
            800.0,    # 800–1000
            1000.0,   # 1000–1200
            1200.0,   # > 1200
            5000.0,
        ]
        
        cmap_elev = ListedColormap(colours)
        norm = BoundaryNorm(bounds, len(colours), clip=True)
        
        # Plot as semi‑transparent background
        elev_plot = ax.pcolormesh(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            cmap=cmap_elev,
            norm=norm,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            # zorder=2,
        )

        # Draw 0 m contour as an accurate coastline
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[0.0],
            colors='black',
            linewidths=0.5,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        # Draw 400 m contour
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[400.0],
            colors='black',
            linewidths=0.3,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

    else:
        raise ValueError('GEBCO DEM returned None')

except Exception as e:
    print(f'Terrain shading failed: {e}')
    print('Falling back to simple land/ocean shading')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
    ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)


# EXPERIMENTAL TOPO SHADING SECTION END
# EXPERIMENTAL TOPO SHADING SECTION END
# EXPERIMENTAL TOPO SHADING SECTION END

# Add MINOR gridlines (tenth degrees) - thin
gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

# Add MID LEVEL gridlines (half degrees) - standard width with labels
gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

# Add MAJOR gridlines (full degrees) - thick with labels
gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees

# Format labels
gl_mid.xformatter = LONGITUDE_FORMATTER
gl_mid.yformatter = LATITUDE_FORMATTER

gl_major.xformatter = LONGITUDE_FORMATTER
gl_major.yformatter = LATITUDE_FORMATTER

# Remove labels from top and right
gl_mid.top_labels = False
gl_mid.right_labels = False
gl_mid.bottom_labels = True
gl_mid.left_labels = True

gl_major.top_labels = False
gl_major.right_labels = False
gl_major.bottom_labels = True
gl_major.left_labels = True

# # Add coastlines ON TOP of radar data
# ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=13)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# CHAD CREATED CYCLICAL COLOUR MAP FOR TIME OF DAY

# ── time-of-day colourbar ─────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(
    cmap = plt.get_cmap('nipy_spectral'),
    norm = plt.Normalize(vmin=0, vmax=1440)   # minutes in a day
)
sm.set_array([])

plt.title('Feature Tracks (dBZ > ' + str(FeatureDBZmin) + ') for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude on ' + \
          RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' UTC')

# ADD THIS TO MAKE THE COLOUR BAR PROPPER SIZE
from mpl_toolkits.axes_grid1 import make_axes_locatable

cax = fig.add_axes([
    ax.get_position().x1 + 0.06,   # left edge (just right of the map)
    ax.get_position().y0,           # bottom edge aligned with map
    0.02,                           # width
    ax.get_position().height        # height exactly matches map
])
# ADD THIS TO MAKE THE COLOUR BAR PROPPER SIZE END

cbar_time = plt.colorbar(sm, cax=cax, orientation='vertical')

# cbar_time = plt.colorbar(sm, ax=ax, orientation='vertical',
#                          pad=0.02, fraction=0.02, shrink=1.0)
cbar_time.set_label('Time of Day (UTC)')

# Tick every hour (every 60 minutes)
hourly_ticks = np.arange(0, 1441, 60)
cbar_time.set_ticks(hourly_ticks)

# Label only every 3 hours, blank string for the rest
tick_labels = [
    f'{int(m // 60):02d}:00' if m % 180 == 0 else ''
    for m in hourly_ticks
]
cbar_time.set_ticklabels(tick_labels)

# Make the hourly tick marks visible but keep 3-hourly labels readable
cbar_time.ax.tick_params(
    which  = 'major',
    length = 4,
    width  = 0.8,
)


plot_feature_tracks(ax, results)


plt.tight_layout()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/FeatureTracks/' + RadarIDno + '/' + RadarFileDate + '/'
                                                                                    
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + str(int(FeatureDBZmin)) + '_FeatureTracks_' + str(Altitude) + 'm.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
plt.close()

In [ ]:
# STRAIGHT LINE TRACKS ON A MAP!

fig, ax = plt.subplots(figsize=(8, 6), 
                       subplot_kw={'projection': ccrs.PlateCarree()})

# gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
# EXPERIMENTAL TOPO SHADING SECTION
# EXPERIMENTAL TOPO SHADING SECTION
# EXPERIMENTAL TOPO SHADING SECTION

# TERRAIN SHADING USING LOCAL GEBCO DEM
lon_min, lon_max = float(xgrid.lon.min()), float(xgrid.lon.max())
lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

try:
    # Path to your GEBCO NetCDF
    gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'

    # Use the helper to get a subset over the radar domain
    dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

    if dem_da is not None:
        # dem_da is an xarray.DataArray with coords lon, lat
        dem_lon = dem_da.lon.values
        dem_lat = dem_da.lat.values
        dem_data = dem_da.values

        # Make 2D lon/lat grids if necessary
        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        # Ensure we have some valid data
        valid = np.isfinite(dem_data)
        if not np.any(valid):
            raise ValueError('DEM has no finite values in this domain')

        colours = [
            '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
        
            '#c4dec2',  # 1: 0–200 m, pale green
            '#e4edc9',  # 2: 200–400 m, greenish-yellow
            '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
            '#e9d7bd',  # 4: 600–800 m, light tan
            '#ddc4aa',  # 5: 800–1000 m, tan
            '#cfb194',  # 6: 1000–1200 m, light brown
            '#b58f6e',  # 7: > 1200 m, darker brown
        ]
        
        bounds = [
            -1000.0,  # ocean below 0
            0.0,      # 0–200
            200.0,    # 200–400
            400.0,    # 400–600
            600.0,    # 600–800
            800.0,    # 800–1000
            1000.0,   # 1000–1200
            1200.0,   # > 1200
            5000.0,
        ]
        
        cmap_elev = ListedColormap(colours)
        norm = BoundaryNorm(bounds, len(colours), clip=True)
        
        # Plot as semi‑transparent background
        elev_plot = ax.pcolormesh(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            cmap=cmap_elev,
            norm=norm,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            # zorder=2,
        )

        # Draw 0 m contour as an accurate coastline
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[0.0],
            colors='black',
            linewidths=0.5,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        # Draw 400 m contour
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[400.0],
            colors='black',
            linewidths=0.3,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

    else:
        raise ValueError('GEBCO DEM returned None')

except Exception as e:
    print(f'Terrain shading failed: {e}')
    print('Falling back to simple land/ocean shading')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
    ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)


# EXPERIMENTAL TOPO SHADING SECTION END
# EXPERIMENTAL TOPO SHADING SECTION END
# EXPERIMENTAL TOPO SHADING SECTION END

# Add MINOR gridlines (tenth degrees) - thin
gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

# Add MID LEVEL gridlines (half degrees) - standard width with labels
gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

# Add MAJOR gridlines (full degrees) - thick with labels
gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees

# Format labels
gl_mid.xformatter = LONGITUDE_FORMATTER
gl_mid.yformatter = LATITUDE_FORMATTER

gl_major.xformatter = LONGITUDE_FORMATTER
gl_major.yformatter = LATITUDE_FORMATTER

# Remove labels from top and right
gl_mid.top_labels = False
gl_mid.right_labels = False
gl_mid.bottom_labels = True
gl_mid.left_labels = True

gl_major.top_labels = False
gl_major.right_labels = False
gl_major.bottom_labels = True
gl_major.left_labels = True

# # Add coastlines ON TOP of radar data
# ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=13)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# ── plot straight-line feature tracks from summary ────────────────────────────
# Colour lines by time of day of birth, using nipy_spectral
track_cmap = plt.get_cmap('nipy_spectral')

for _, row in summary.iterrows():
    # Skip single-frame features (no displacement)
    if row['n_timesteps'] < 2:
        continue

    # Colour by birth time of day in minutes
    dt         = pd.Timestamp(row['birth_t'])
    minutes    = dt.hour * 60 + dt.minute + dt.second / 60.0
    normalised = minutes / 1440.0
    colour     = track_cmap(normalised)

    ax.plot(
        [row['birth_lon'], row['death_lon']],
        [row['birth_lat'], row['death_lat']],
        color          = colour,
        linewidth      = 1.0,
        transform      = ccrs.PlateCarree(),
        zorder         = 20,
        alpha          = 0.7,
        solid_capstyle = 'round',
    )

    # Small dot at birth location
    ax.scatter(
        row['birth_lon'], row['birth_lat'],
        s         = 8,
        color     = colour,
        transform = ccrs.PlateCarree(),
        zorder    = 21,
        alpha     = 0.7,
    )

# ADD THIS TO MAKE THE COLOUR BAR PROPPER SIZE
from mpl_toolkits.axes_grid1 import make_axes_locatable

cax = fig.add_axes([
    ax.get_position().x1 + 0.02,   # left edge (just right of the map)
    ax.get_position().y0,           # bottom edge aligned with map
    0.02,                           # width
    ax.get_position().height        # height exactly matches map
])
# ADD THIS TO MAKE THE COLOUR BAR PROPPER SIZE END

cbar_time = plt.colorbar(sm, cax=cax, orientation='vertical')

# cbar_time = plt.colorbar(sm, ax=ax, orientation='vertical',
#                          pad=0.02, fraction=0.02, shrink=1.0)
cbar_time.set_label('Time of Day at Birth (UTC)')

# Tick every hour (every 60 minutes)
hourly_ticks = np.arange(0, 1441, 60)
cbar_time.set_ticks(hourly_ticks)

# Label only every 3 hours, blank string for the rest
tick_labels = [
    f'{int(m // 60):02d}:00' if m % 180 == 0 else ''
    for m in hourly_ticks
]
cbar_time.set_ticklabels(tick_labels)

# Make the hourly tick marks visible but keep 3-hourly labels readable
cbar_time.ax.tick_params(
    which  = 'major',
    length = 4,
    width  = 0.8,
)

hourly_ticks = np.arange(0, 1441, 60)
cbar_tracks.set_ticks(hourly_ticks)
tick_labels = [
    f'{int(m // 60):02d}:00' if m % 180 == 0 else ''
    for m in hourly_ticks
]
cbar_tracks.set_ticklabels(tick_labels)
cbar_tracks.ax.tick_params(which='major', length=4, width=0.8)


In [ ]:
# OUTDATED SINGLE TIME STEP VERSION
# OUTDATED SINGLE TIME STEP VERSION
# OUTDATED SINGLE TIME STEP VERSION
# OUTDATED SINGLE TIME STEP VERSION
# OUTDATED SINGLE TIME STEP VERSION
# OUTDATED SINGLE TIME STEP VERSION

import numpy as np
from scipy.ndimage import label
import pandas as pd

def find_reflectivity_features(xgrid, time_index=0, level_index=4, threshold_dbz=20.0):
    """
    Find continuous regions of radar reflectivity above a given threshold.

    Parameters
    ----------
    xgrid : xarray.Dataset
        The radar grid dataset.
    time_index : int
        Index along the time dimension.
    level_index : int
        Index along the vertical/level dimension.
    threshold_dbz : float
        Reflectivity threshold in dBZ. Only values strictly above this
        are included in features.

    Returns
    -------
    features : list of dict
        Each dict contains:
            - 'feature_id'        : int, label assigned to this feature
            - 'n_cells'           : int, number of grid cells in the feature
            - 'area_km2'          : float, area of the feature in km^2
            - 'mean_dbz'          : float, mean reflectivity of the feature (dBZ)
            - 'centre_x_km'       : float, reflectivity-weighted centre of gravity (km)
            - 'centre_y_km'       : float, reflectivity-weighted centre of gravity (km)
            - 'refl_mass_dBZkm2'  : float, integrated reflectivity mass in dB(Z km^2)
    labelled_grid : np.ndarray (301 x 301)
        Integer array where each feature is labelled 1..N, and 0 is background.
    """

    # --- Extract the 2D reflectivity slice ---
    refl = xgrid['corrected_reflectivity'][time_index][level_index].values  # (301, 301)

    # --- Extract coordinate arrays, converting from metres to kilometres ---
    x_coords = xgrid['x'].values / 1000.0  # shape (301,) — km relative to radar, West-East
    y_coords = xgrid['y'].values / 1000.0  # shape (301,) — km relative to radar, South-North

    # --- Grid cell area (1 km spacing, so each cell is 1 km^2) ---
    cell_area_km2 = (x_coords[1] - x_coords[0]) * (y_coords[1] - y_coords[0])

    # Build 2D meshgrids so every cell has an (x, y) coordinate
    x_grid_2d, y_grid_2d = np.meshgrid(x_coords, y_coords)

    # --- Create binary mask of cells above threshold ---
    mask = refl > threshold_dbz  # boolean (301, 301)

    # --- Label connected regions ---
    structure_8conn = np.ones((3, 3), dtype=int)  # 8-connectivity
    labelled_grid, n_features = label(mask, structure=structure_8conn)

    print(f"Found {n_features} feature(s) above {threshold_dbz} dBZ.\n")

    # --- Compute statistics for each feature ---
    features = []

    for feature_id in range(1, n_features + 1):
        feature_mask = labelled_grid == feature_id
        refl_values = refl[feature_mask]

        n_cells   = int(feature_mask.sum())
        area_km2  = float(n_cells * cell_area_km2)
        mean_dbz  = float(refl_values.mean())

        # Linear reflectivity values (Z, not dBZ)
        linear_z = 10.0 ** (refl_values / 10.0)

        # Centre of gravity weighted by linear reflectivity
        total_weight = linear_z.sum()
        centre_x = float((x_grid_2d[feature_mask] * linear_z).sum() / total_weight)
        centre_y = float((y_grid_2d[feature_mask] * linear_z).sum() / total_weight)

        # Reflectivity mass: sum of Z * cell_area, converted to dB(Z km^2)
        refl_mass_linear = float((linear_z * cell_area_km2).sum())  # Z km^2
        refl_mass_dB     = float(10.0 * np.log10(refl_mass_linear)) # dB(Z km^2)

        features.append({
            'feature_id'       : feature_id,
            'n_cells'          : n_cells,
            'area_km2'         : round(area_km2, 2),
            'mean_dbz'         : round(mean_dbz, 2),
            'centre_x_km'      : round(centre_x, 3),
            'centre_y_km'      : round(centre_y, 3),
            'refl_mass_dBZkm2' : round(refl_mass_dB, 2),
        })

    # --- Print results as a formatted table ---
    df = pd.DataFrame(features)
    df.columns = [
        'Feature ID', 'N Cells', 'Area (km²)',
        'Mean dBZ', 'Centre X (km)', 'Centre Y (km)',
        'Refl. Mass (dB(Z km²))'
    ]
    print(df.to_string(index=False))

    return features, labelled_grid

In [ ]:
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE COMPRESSED OR UNCOMRESSED FILES
CompBool = 1     # 1 for yes compressed, 0 for no, uncompressed files

# # CHOOSE THE RADAR
# RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
# RadarMonth = 2
# RadarDay   = 14
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# CHOOSE YOUR ALTITUDE
Altitude = 2000  # [m] choose a multiple of 500 m to look at a CAPI for

# CHOOSE YOUR VARIABLE
Var = 'Z'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# CHOOSE YOUR QUALITY CONTROL SETTINGS
# taken from Aragon et al. 2024
MinValidZDR = -4 # NO VALID DATA TO USE THE OPTION YET
MaxValidZDR =  4 # NO VALID DATA TO USE THE OPTION YET
MinValidRhoHV = 0.85





# USER CHOICE FOLLOW-ON SECTION

# compression choice follow-on (select the folder to load from)
if (CompBool):
    RadarGridsFolder = 'CompressedRadarGrids'
else:
    RadarGridsFolder = 'RadarGrids'  

# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'Horz'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
    #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


    NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                             + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
    # try to load in the netcdf file and if it doesn't work, just keep going through the loop
    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # index in the netcdf altitude variable for the altitude you want
    alti = np.where(xgrid.z == Altitude) # this is a double nested array for some reason
    alti = alti[0][0] # take the index out of the double nested array

    # quit out if the altitude does not correspond to one in the netCDF file
    if ( np.size(alti) != 1): 
        raise ValueError( str(Altitude) + ' m is not a valid altitude in the data')


    # ROUGH QUALITY CONTROL SECTION
    # create a mask only where these quality control condtions are met
    ConditionGridA = xgrid['corrected_differential_reflectivity'] > MinValidZDR   # (y, x) boolean masks
    
    # I WOULD LIKE TO ADD CONDITIONS WITH DIFFERENTIAL REFLECTIVITY, BUT THIS RADAR DOES NOT HAVE VALID DATA YET
    # ConditionGridB = xgrid['corrected_differential_reflectivity'] < MaxValidZDR  
    # ConditionGridC = xgrid['corrected_cross_correlation_ratio'] > MinValidRhoHV
    
    ConditionGrid = ConditionGridA #* ConditionGridB * ConditionGridC   # combined boolean mask



    fig, ax = plt.subplots(figsize=(8, 6), 
                           subplot_kw={'projection': ccrs.PlateCarree()})

    # gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    
    # TERRAIN SHADING USING LOCAL GEBCO DEM
    lon_min, lon_max = float(xgrid.lon.min()), float(xgrid.lon.max())
    lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

    try:
        # Path to your GEBCO NetCDF
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'

        # Use the helper to get a subset over the radar domain
        dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

        if dem_da is not None:
            # dem_da is an xarray.DataArray with coords lon, lat
            dem_lon = dem_da.lon.values
            dem_lat = dem_da.lat.values
            dem_data = dem_da.values

            # Make 2D lon/lat grids if necessary
            if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
            else:
                dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

            # Ensure we have some valid data
            valid = np.isfinite(dem_data)
            if not np.any(valid):
                raise ValueError('DEM has no finite values in this domain')

            colours = [
                '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
            
                '#c4dec2',  # 1: 0–200 m, pale green
                '#e4edc9',  # 2: 200–400 m, greenish-yellow
                '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
                '#e9d7bd',  # 4: 600–800 m, light tan
                '#ddc4aa',  # 5: 800–1000 m, tan
                '#cfb194',  # 6: 1000–1200 m, light brown
                '#b58f6e',  # 7: > 1200 m, darker brown
            ]
            
            bounds = [
                -1000.0,  # ocean below 0
                0.0,      # 0–200
                200.0,    # 200–400
                400.0,    # 400–600
                600.0,    # 600–800
                800.0,    # 800–1000
                1000.0,   # 1000–1200
                1200.0,   # > 1200
                5000.0,
            ]
            
            cmap_elev = ListedColormap(colours)
            norm = BoundaryNorm(bounds, len(colours), clip=True)
            
            # Plot as semi‑transparent background
            elev_plot = ax.pcolormesh(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                cmap=cmap_elev,
                norm=norm,
                alpha=1.0,
                transform=ccrs.PlateCarree(),
                # zorder=2,
            )

            # Draw 0 m contour as an accurate coastline
            coast_contour = ax.contour(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                levels=[0.0],
                colors='black',
                linewidths=0.5,
                transform=ccrs.PlateCarree(),
                zorder=15,  # above radar and topo
            )

            # Draw 400 m contour
            coast_contour = ax.contour(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                levels=[400.0],
                colors='black',
                linewidths=0.3,
                transform=ccrs.PlateCarree(),
                zorder=15,  # above radar and topo
            )

            print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

        else:
            raise ValueError('GEBCO DEM returned None')

    except Exception as e:
        print(f'Terrain shading failed: {e}')
        print('Falling back to simple land/ocean shading')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)


    # EXPERIMENTAL TOPO SHADING SECTION END
    # EXPERIMENTAL TOPO SHADING SECTION END
    # EXPERIMENTAL TOPO SHADING SECTION END

    # apply the condtional mask to the variable array before plotting
    ValidVariableArray = xgrid[VarNameLong].where(ConditionGrid)
    PlottingArray = ValidVariableArray[0,alti,:,:]

    GridViewer = ax.pcolormesh(xgrid.lon, xgrid.lat, PlottingArray, 
                               cmap=VarColourBar , norm=VarColourBar_norm, transform=ccrs.PlateCarree())

    # ── plot feature centres of mass for this timestep ────────────────────────
    # Subtract 4min 30sec from each results timestamp and round to nearest 5 min
    # e.g. 02:04:30 - 00:04:30 = 02:00:00 → matches MinOfDay=120
    result_timestamps = list(results.keys())

    # Build a lookup: rounded_minute_of_day → original timestamp
    rounded_lookup = {}
    for ts in result_timestamps:
        dt          = pd.Timestamp(ts)
        adjusted    = dt - pd.Timedelta(minutes=4, seconds=30)
        # Round to nearest 5 minutes
        total_mins  = adjusted.hour * 60 + adjusted.minute
        rounded_min = round(total_mins / 5) * 5
        rounded_lookup[rounded_min] = ts

    if MinOfDay in rounded_lookup:
        df_this_t = results[rounded_lookup[MinOfDay]]
        if not df_this_t.empty:
            ax.scatter(
                df_this_t['centre_lon'].values,
                df_this_t['centre_lat'].values,
                s         = 10,
                color     = [1,0,1], # magenta
                transform = ccrs.PlateCarree(),
                zorder    = 25,
            )


    cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.ax.set_ylim(-VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, 10))  # tick every 10 dBZ

    # Add MINOR gridlines (tenth degrees) - thin
    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
    gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

    # Add MID LEVEL gridlines (half degrees) - standard width with labels
    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
    gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

    # Add MAJOR gridlines (full degrees) - thick with labels
    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    
    # Format labels
    gl_mid.xformatter = LONGITUDE_FORMATTER
    gl_mid.yformatter = LATITUDE_FORMATTER
    
    gl_major.xformatter = LONGITUDE_FORMATTER
    gl_major.yformatter = LATITUDE_FORMATTER
    
    # Remove labels from top and right
    gl_mid.top_labels = False
    gl_mid.right_labels = False
    gl_mid.bottom_labels = True
    gl_mid.left_labels = True
    
    gl_major.top_labels = False
    gl_major.right_labels = False
    gl_major.bottom_labels = True
    gl_major.left_labels = True

    # # Add coastlines ON TOP of radar data
    # ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=13)
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    plt.title(VarName + ' (ρHV > ' + str(MinValidRhoHV) + ') for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')

    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/' + \
                                                                                           VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + 'FeatureDots/'
                                                                                                      # RhoHV 0.85 becomes 85 in file name
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
                 PlotType + str(Altitude) + 'm.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('doing')
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    plt.close()

In [ ]:
SavePath

In [ ]:
# GIF MAKER
# FOR Horizontal Cross Sections

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/' + \
                                                                                       VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + 'FeatureDots/'
                                                                                                  # RhoHV 0.85 becomes 85 in file name         
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named

images = [Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(PlotType + str(Altitude) + 'm.png')]


GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Horz/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarName + '_RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '_' + str(Altitude) + 'mFeatureDots.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

# make a folder to store the new GIF in if one does not exist already
if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(GIFsavePath, save_all=True, append_images=images[1:], duration=200, loop=0)          
                                                                   # ms per frame       0 = loop forever
print('Saved GIF for ' + RadarFileDate)

In [ ]:
SavedFolder 

In [ ]:
summary